# Bloc 4 — Faire tourner wyat (19 min)

**Situation.** Le modèle est branché sur le jeu **dev**. Une seule modification de texte, l'option facturation raccourcie en *« Facturation : questions sur les factures »*, suffit à envoyer *« vous m'avez débité 9,99 € »* chez *autre*. Personne n'a touché au modèle.

Vous n'avez qu'à **exécuter les cellules dans l'ordre** (Maj+Entrée). On travaille sur le jeu **dev** (63 messages synthétiques étiquetés ; 20 en *mode rapide* sur un PC sans GPU) : ici on a le droit de regarder les erreurs et de modifier les options.

Une barre de progression et une durée estimée s'affichent pendant les calculs. Une décision déjà calculée sur votre ordinateur n'est pas recalculée.

In [ ]:
# Préparation (rien à modifier) — exécutez avec Maj+Entrée
import sys; sys.path.insert(0, "..")   # le paquet atelier/ (installé par installer.sh ; ceci au cas où)
import matplotlib.pyplot as plt
import pandas as pd
import wyat
import atelier as at                  # même boîte à outils que la commande `atelier` et les exemples/
pd.set_option("display.max_colwidth", 90)
modele = at.charger_modele()

## 1. Les options actuelles (le « contrat » donné au modèle)

In [ ]:
pd.Series(at.OPTIONS, name="description lue par le modèle")

## 2. On lance le tri sur les messages de dev

In [ ]:
dev = at.lire("dev.csv")
rapport, res = at.evaluer(modele, dev, at.OPTIONS)
print(f"Exactitude (étiquette de référence) : {res.correct.mean():.1%}  ({res.correct.sum()}/{len(res)})")
print(f"Exactitude (étiquette « acceptable ») : {res.acceptable.mean():.1%}")
res.head(10)

## 3. Où se trompe-t-il ? (matrice de confusion : lignes = vérité, colonnes = modèle)

In [ ]:
pd.crosstab(res.label, res.prediction, margins=True)

In [ ]:
print("Exactitude par difficulté :")
display(res.groupby("difficulte").correct.agg(["mean", "count"]).round(2))
print("Les erreurs, de la plus « sûre » à la plus hésitante :")
res[~res.correct].sort_values("marge", ascending=False)[["message", "label", "prediction", "2e_choix", "marge", "difficulte"]]

## 4. Les marges : le modèle sait-il quand il hésite ?
**Question** : regardez la colonne `marge` des erreurs ci-dessus, puis celle des réponses justes (graphique ci-dessous). Que remarquez-vous ? Qu'en feriez-vous, si vous étiez responsable du support ?
*Réponse attendue* : les erreurs ont surtout des marges faibles. Idée : **quand la marge est faible, on passe la main à un humain**. C'est la politique du bloc 5.

In [ ]:
plt.figure(figsize=(7, 3))
plt.hist([res[res.correct].marge, res[~res.correct].marge], bins=10, range=(0, 1), stacked=True,
         label=["correct", "erreur"], color=["#2a9d8f", "#e76f51"])
plt.xlabel("marge (proba 1er − proba 2e)"); plt.ylabel("messages"); plt.legend(); plt.show()
print("Marge médiane — correct :", res[res.correct].marge.median(), "| erreur :", res[~res.correct].marge.median())

## 5. Exercice : réécrire une option
Voici une variante où l'option *facturation* dit seulement **« Facturation : questions sur les factures »**. Elle ne parle plus des **débits / prélèvements**. Lançons-la :

In [ ]:
rapport_ex, res_ex = at.evaluer(modele, dev, at.OPTIONS_EXERCICE)
print(f"Exactitude avec la variante : {res_ex.correct.mean():.1%}  ({res_ex.correct.sum()}/{len(res_ex)})")
res_ex[~res_ex.correct & (res_ex.label == "facturation")][["message", "prediction", "marge"]]

**À vous !** Modifiez le texte de l'option `facturation` ci-dessous (ou une autre option qui vous semble mal écrite), puis exécutez la cellule. Règles du jeu :
- décrivez ce que l'option **couvre**, comme pour un collègue qui découvre le service ;
- utilisez les **mots des clients** (« prélevé », « débité », « remboursé ») ;
- n'ajoutez pas d'option « tout le reste / pas sûr » : elle aspire les messages.

In [ ]:
mes_options = dict(at.OPTIONS_EXERCICE)
mes_options["facturation"] = "Facturation : questions sur les factures"   # <-- réécrivez ce texte
rapport_moi, res_moi = at.evaluer(modele, dev, mes_options)
print(f"Avant : {res_ex.correct.mean():.1%}   →   Après votre réécriture : {res_moi.correct.mean():.1%}")

### Solution proposée (à dérouler après l'exercice)

In [ ]:
solution = dict(at.OPTIONS_EXERCICE)
solution["facturation"] = ("Facturation et paiements : montant d'une facture, débits ou prélèvements bancaires, "
                           "remboursement, moyen ou date de paiement")
rapport_sol, res_sol = at.evaluer(modele, dev, solution)
comparaison = pd.DataFrame({"variante (factures seulement)": res_ex.set_index("id").prediction,
                            "solution (débits/prélèvements)": res_sol.set_index("id").prediction,
                            "référence": res_sol.set_index("id").label,
                            "message": res_sol.set_index("id").message})
changes = comparaison[comparaison.iloc[:, 0] != comparaison.iloc[:, 1]]
print(f"Variante : {res_ex.correct.mean():.1%}  →  solution : {res_sol.correct.mean():.1%}   ({len(changes)} messages ont changé de service)")
changes

### Contre-exemple : des libellés d'un mot + un attrape-tout

In [ ]:
rapport_ft, res_ft = at.evaluer(modele, dev, at.OPTIONS_FOURRE_TOUT)
print(f"Libellés d'un mot + « Autre, pas sûr, tout le reste » : {res_ft.correct.mean():.1%}")
print("Prédictions :", res_ft.prediction.value_counts().to_dict())
print("Vérité      :", res_ft.label.value_counts().to_dict())

**À retenir** : le libellé d'une option *est* sa définition pour le modèle. On réécrit les options **sur dev**, jamais sur le test scellé.

---
**Étape 4, point de contrôle** (`missions/4_reparer_option/MISSION.md`) : le même objectif, à atteindre comme vous voulez (ce notebook, la commande `atelier`, ou un script).
Vérification sur dev : `atelier verifier 4` dans un terminal, ou la cellule ci-dessous.

In [ ]:
from pathlib import Path
from atelier.missions import preparer, verifier
preparer(Path("../travail"))   # crée travail/ avec les fichiers de départ s'ils manquent (n'écrase rien)
print(verifier(4, globals().get("modele"), Path("../travail")).texte())   # réutilise le modèle déjà chargé